# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, getpass
import duckdb

# Colab: use the Secrets panel (key icon) to set HF_TOKEN, then userdata.get() picks it up.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Point straight at the March partition -- this is a mid-panel month, not the
# sealed final month (2026-06) and not the _sample table (that IS the final month).
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# DESCRIBE touches Parquet metadata/schema only -- near-free, and it tells us the
# real column names before we write a single WHERE clause against them.
con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_march']}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**In plain words:**

1. **One row means:** one content item's daily search-performance record — the triple
   `(report_date, client_hash_id, content_hash_id)` inside `fact_content_daily_performance`.
   My lane (Refresh / Content Opportunity Scoring) rolls this daily grain up to **one row
   per content item, summarized over one month** for scoring — but the raw table's grain,
   which I verify below, is the daily one.
2. **Table(s) I use:** `fact_content_daily_performance` (the `month=2026-03` partition only)
   for the time series; `dim_content` for content-level context (content age); `dim_clients`
   only to confirm which clients have GA4 coverage in this window.
3. **Time window:** calendar month **2026-03-01 through 2026-03-31** — a mid-panel month.
   I never touch `fact_content_daily_performance_sample` or the `2026-06` partition for label
   logic, since the sample table *is* the final month and would silently seal in the outcome
   window.
4. **What I'd predict/rank (label or proxy):** a proxy label, `is_declining_proxy` — 1 when a
   content item's GSC impressions in the second half of March (16th–31st) fall below 80% of
   its first-half impressions (1st–15th), else 0. This mirrors the starter dataset's
   `trend_direction` idea, but built myself from raw daily rows instead of trusting a
   pre-computed column. Used to rank content items into a review queue.
5. **What I deliberately exclude:** FlyRank's own product-decision fields (`health_score`,
   `priority_score`, `action_type`, `refresh_tier`) — they aren't shipped in this release, and
   I exclude them on principle: feeding a product decision into the model as a feature or
   label would just teach it to reproduce that decision, not discover anything (a circular
   result). I also exclude `keyword_hash_id` / `url_hash_id` as anything other than grouping
   keys — they're pseudonyms, not signal.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:

import pandas as pd

field_map = pd.DataFrame([
    ("report_date",       "context",  "defines the time window / the h1-vs-h2 split; never a model input"),
    ("client_hash_id",    "context",  "grouping/joining only -- a pseudonym, carries no signal"),
    ("content_hash_id",   "context",  "grouping/joining only -- a pseudonym, carries no signal"),
    ("gsc_impressions",   "feature",  "observed daily signal, known the moment the day closes"),
    ("gsc_clicks",        "feature",  "observed daily signal, known the moment the day closes"),
    ("gsc_avg_position",  "feature",  "observed daily signal, known the moment the day closes"),
    ("ga4_data_available","context",  "an availability flag used to filter rows, not to predict from"),
    ("is_declining_proxy","label",    "the thing I predict -- built from h1 vs h2 impressions, never a feature"),
    ("content_created_at","excluded", "from dim_content; excluded here -- content-age effects are a fine follow-up but out of scope for this notebook's 5-feature limit"),
    ("health_score / priority_score / action_type / refresh_tier", "excluded", "FlyRank product decisions, not shipped, and circular if ever rebuilt and reused as a label"),
])
field_map.columns = ["field", "bucket", "why"]
field_map

,field,bucket,why
0,report_date,context,defines the time window / the h1-vs-h2 split; ...
1,client_hash_id,context,"grouping/joining only -- a pseudonym, carries ..."
2,content_hash_id,context,"grouping/joining only -- a pseudonym, carries ..."
3,gsc_impressions,feature,"observed daily signal, known the moment the da..."
4,gsc_clicks,feature,"observed daily signal, known the moment the da..."
5,gsc_avg_position,feature,"observed daily signal, known the moment the da..."
6,ga4_data_available,context,"an availability flag used to filter rows, not ..."
7,is_declining_proxy,label,the thing I predict -- built from h1 vs h2 imp...
8,content_created_at,excluded,from dim_content; excluded here -- content-age...
9,health_score / priority_score / action_type / ...,excluded,"FlyRank product decisions, not shipped, and ci..."


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 of 3 — grain: is one row really `(report_date, client_hash_id, content_hash_id)`?

In [3]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"duplicate-grain rows found: {len(grain_check)} (0 means the grain holds)")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate-grain rows found: 0 (0 means the grain holds)


,report_date,client_hash_id,content_hash_id,n


### Query 2 of 3 — my slice's row count and date span

In [4]:
span_check = con.sql(f"""
    SELECT
        COUNT(*)                      AS total_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        MIN(report_date)              AS min_date,
        MAX(report_date)              AS max_date
    FROM {TABLES['fact_march']}
""").df()

span_check

,total_rows,distinct_content_items,distinct_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


### Query 3 of 3 — availability: filter with `IS TRUE`, how many rows survive?

Rows before a client's GA4 start are zero-filled, not zero-engagement — the flag is what
tells the two apart.

In [5]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*)                                            AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_march']}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


## Five features (max) — built from the same March slice

Decision moment: **end of day March 15**. Every feature below is summed/averaged only over
`report_date BETWEEN '2026-03-01' AND '2026-03-15'` (the first half of the month) — strictly
before the March 16–31 window my proxy label is computed from. That split is what keeps the
features honest: none of them can see into the label window.

In [6]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS imp_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)               AS ctr_h1,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)       AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10   -- minimum volume, so ctr/position aren't noise
""").df()

print(f"{len(feature_frame):,} content items with enough March 1-15 volume")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items with enough March 1-15 volume


,client_hash_id,content_hash_id,imp_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,0.004662,4.247255,15
1,client_73cda7b4e4f265ea,content_05597932fe4da067,18.0,0.0,0.000000,9.055556,11
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,0.0,0.000000,3.763426,15
3,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,0.001592,5.330069,15
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,0.007031,4.468441,15


**Every feature, one line each — knowable at the decision moment because…**

1. `imp_h1` (total GSC impressions, Mar 1–15) — knowable because it only sums days that have
   already closed before the March 16 decision point.
2. `clicks_h1` (total GSC clicks, Mar 1–15) — same reasoning: a same-window daily sum.
3. `ctr_h1` (`clicks_h1 / imp_h1`) — a ratio of two numbers that are themselves only from the
   h1 window, so it's knowable the moment h1 closes.
4. `avg_position_h1` (mean GSC position, Mar 1–15, positions > 0 only) — an observed daily
   measurement averaged over days that have already happened.
5. `active_days_h1` (distinct days with impressions > 0, Mar 1–15) — a count of past days;
   cannot change once March 15 ends.

## The trap: add one label-derived column on purpose

The label (`is_declining_proxy`) is built from second-half impressions (`imp_h2`) versus
`imp_h1`. If `imp_h2` — or anything computed from it — ever sits in the feature table, the
model isn't predicting decline, it's just reading the answer back off the label's own input.
Watch the score jump when I do exactly that, then remove it.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

label_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31') AS imp_h2
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2
""").df()

leak_demo = feature_frame.merge(label_frame, on=['client_hash_id', 'content_hash_id'], how='inner')
leak_demo['is_declining_proxy'] = (leak_demo['imp_h2'] < 0.8 * leak_demo['imp_h1']).astype(int)
leak_demo = leak_demo.dropna(subset=['ctr_h1', 'avg_position_h1'])

honest_features = ['imp_h1', 'clicks_h1', 'ctr_h1', 'avg_position_h1', 'active_days_h1']
leaky_features  = honest_features + ['imp_h2']   # the trap: the label's own input, smuggled in as a "feature"

def quick_auc(cols):
    X = leak_demo[cols]
    y = leak_demo['is_declining_proxy']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

print(f"honest ROC-AUC (5 real features):        {quick_auc(honest_features):.3f}")
print(f"leaky  ROC-AUC (5 features + imp_h2):     {quick_auc(leaky_features):.3f}  <- jumps toward 1.0")
print()
print("imp_h2 is literally the numerator the label was built from -- of course the leaky")
print("model 'predicts' it almost perfectly. Deleting it and keeping the honest number")
print("is the whole lesson from notebook 02, now shown on real warehouse data.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

honest ROC-AUC (5 real features):        0.582
leaky  ROC-AUC (5 features + imp_h2):     1.000  <- jumps toward 1.0

imp_h2 is literally the numerator the label was built from -- of course the leaky
model 'predicts' it almost perfectly. Deleting it and keeping the honest number
is the whole lesson from notebook 02, now shown on real warehouse data.


**The leak is now deleted.** `leaky_features` (with `imp_h2`) was only ever used for this
demonstration above — it is never carried into the feature frame, never saved to
`work/outputs/`, and never reused in later notebooks. The number I keep going forward is the
**honest ROC-AUC** printed above, from `honest_features` only.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
limits_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_clients,
        COUNT(*) FILTER (WHERE gsc_data_start > DATE '2026-03-01') AS clients_without_full_march_gsc_history,
        COUNT(*) FILTER (WHERE ga4_data_start IS NULL OR ga4_data_start > DATE '2026-03-01') AS clients_without_full_march_ga4_history
    FROM {TABLES['dim_clients']}
""").df()

limits_check

,march_clients,clients_without_full_march_gsc_history,clients_without_full_march_ga4_history
0,104,15,78


**Named limitation:** this is a single 31-day window on an **unbalanced panel** — some
clients' GSC or GA4 history starts partway through, or after, March 2026 (counted above), so
their March rows understate real activity rather than reflecting zero activity. A one-month
proxy label also cannot rule out seasonality or consolidation (a sibling page absorbing the
traffic) as the real cause of a "decline" — separating those from a genuine decline needs a
longer window and the checks described in the `hunting-leakage-and-validating` skill, which is
out of scope for this notebook.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.